# [xihanzu-NR] HydraScript Qwen3-0.6B V3 (6K .hyx + 965 .hys)

- Dataset: 6,965 sample (UI .hyx + Backend Logic .hys 38 segmen)
- Base Model: Qwen/Qwen3-0.6B (BF16, No Quantization)
- LoRA: r=16, alpha=32, target=all-linear
- Batch: per_device=1, grad_accum=16 (effective 16)
- Auto-download dataset langsung dari VPS via curl
- Output akhir: GGUF Q8_0 quantize siap pakai di CPU VPS


In [ ]:
# [xihanzu-NR] Cell 1: Environment & Dependencies
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
!pip install -q -U transformers datasets peft accelerate
!pip uninstall -y torchao 2>/dev/null || true
print("Dependencies installed!")


In [ ]:
# [xihanzu-NR] Cell 2: Download Dataset V3 from VPS
import os, urllib.request
DATA_URL = "http://103.74.5.133:8088/hydrascript_dataset_v3.jsonl"
LOCAL_PATH = "/kaggle/working/hydrascript_dataset_v3.jsonl"

print(f"Downloading dataset from {DATA_URL}...")
urllib.request.urlretrieve(DATA_URL, LOCAL_PATH)
size_mb = os.path.getsize(LOCAL_PATH) / (1024 * 1024)
with open(LOCAL_PATH, "r", encoding="utf-8") as f:
    line_count = sum(1 for line in f if line.strip())
print(f"Downloaded {line_count} samples ({size_mb:.2f} MB) successfully!")


In [ ]:
# [xihanzu-NR] Cell 3: Load Model & Tokenizer (BF16, No Quant)
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = "Qwen/Qwen3-0.6B"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading base model (BF16)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
print(f"VRAM after model load: {torch.cuda.memory_allocated()/1e9:.2f} GB")


In [ ]:
# [xihanzu-NR] Cell 4: Tokenize with Response-Only Masking
import json
from datasets import Dataset

RESPONSE_KEY = "### Response:
"
MAX_SEQ_LEN = 1024

data = []
with open(LOCAL_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            d = json.loads(line)
            data.append({"text": d["formatted"]})

raw = Dataset.from_list(data).train_test_split(test_size=0.05, seed=42)
print(f"Train: {len(raw["train"])} | Val: {len(raw["test"])}")

def tokenize_and_mask(examples):
    all_input_ids, all_labels, all_attn = [], [], []
    for text in examples["text"]:
        tok = tokenizer(text, truncation=True, max_length=MAX_SEQ_LEN,
                        padding=False, add_special_tokens=True)
        input_ids = tok["input_ids"]
        labels = list(input_ids)
        if RESPONSE_KEY in text:
            prompt_part = text.split(RESPONSE_KEY)[0] + RESPONSE_KEY
            p_len = len(tokenizer(prompt_part, truncation=True,
                                  max_length=MAX_SEQ_LEN, padding=False,
                                  add_special_tokens=True)["input_ids"])
            for i in range(min(p_len, len(labels))):
                labels[i] = -100
        all_input_ids.append(input_ids)
        all_labels.append(labels)
        all_attn.append(tok["attention_mask"])
    return {"input_ids": all_input_ids, "labels": all_labels,
            "attention_mask": all_attn}

print("Tokenizing dataset...")
tokenized = raw.map(tokenize_and_mask, batched=True, batch_size=500,
                    remove_columns=["text"], desc="Tokenizing")

sl = tokenized["train"][0]["labels"]
n_masked = sum(1 for x in sl if x == -100)
n_active = sum(1 for x in sl if x != -100)
print(f"Sanity check: {n_masked} prompt tokens masked / {n_active} active response tokens")
assert n_active > 0, "BUG: all tokens masked"


In [ ]:
# [xihanzu-NR] Cell 5: Train LoRA (batch=1, grad_accum=16, 3 epochs)
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model, padding=True,
    pad_to_multiple_of=8, label_pad_token_id=-100,
)

eff_batch = 1 * 16
total_steps = (len(tokenized["train"]) // eff_batch) * 3
warmup = max(10, int(total_steps * 0.05))
print(f"Total training steps: ~{total_steps}, Warmup: {warmup}")

args = TrainingArguments(
    output_dir="./qwen3_hydra_v3_out",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    num_train_epochs=3,
    logging_steps=15,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=150,
    save_total_limit=2,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    warmup_steps=warmup,
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",
    dataloader_pin_memory=False,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=data_collator,
    args=args,
)

print("=== START TRAINING ===")
trainer.train()
print("=== TRAINING COMPLETE ===")


In [ ]:
# [xihanzu-NR] Cell 6: Merge LoRA to Base & Convert to GGUF Q8_0
import subprocess
print("Merging LoRA weights into base model...")
merged = model.merge_and_unload()
MERGED_DIR = "/kaggle/working/qn3_hxprjkt_merged"
merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged model saved to {MERGED_DIR}")

print("Cloning llama.cpp for GGUF conversion...")
!git clone --depth 1 https://github.com/ggerganov/llama.cpp /kaggle/working/llama.cpp
!pip install -q -r /kaggle/working/llama.cpp/requirements.txt

GGUF_OUT = "/kaggle/working/qn3-hxprjkt-q8_0.gguf"
print(f"Converting merged model to GGUF Q8_0: {GGUF_OUT}...")
!python3 /kaggle/working/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {GGUF_OUT} --outtype q8_0

import os
if os.path.exists(GGUF_OUT):
    sz = os.path.getsize(GGUF_OUT) / (1024 * 1024)
    print(f"SUCCESS! GGUF ready: {GGUF_OUT} ({sz:.2f} MB)")
else:
    print("GGUF conversion finished. Checking files in working dir:")
    !ls -lh /kaggle/working/
